# USD/IDR: GARCH-family dan forecasting arah

## tl;dr
GARCH, EGARCH, dan GJR-GARCH dievaluasi sebagai forecaster **varians**, bukan arah. Arah USD/IDR adalah tugas klasifikasi terpisah dengan baseline persistence-sign dan Logistic Regression. Semua hasil berasal dari one-step-ahead test set kronologis.

## Context & Methods

- Train/test: 80/20 secara kronologis; test tidak ikut memilih spesifikasi.
- Volatilitas: inovasi Student-t; parameter direfit setiap 63 observasi, lalu setiap forecast memakai history yang tersedia sebelum return target.
- Metrik volatilitas: MAE, RMSE, dan QLIKE terhadap squared return (%)². Squared return adalah proxy yang noisy.
- Arah: hanya fitur yang tersedia sebelum target (lag return, lag volatilitas, lag perubahan US10Y, dan lag filtered regime probability).

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.forecasting import run_experiment
RAW_PATH = PROJECT_ROOT / 'data/raw/yahoo_usd_idr_us10y.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'

## Results

Pipeline membaca snapshot lokal, menjalankan model, dan menyimpan seluruh tabel serta chart ke `outputs/`.

In [2]:
result = run_experiment(RAW_PATH, OUTPUT_DIR)
print('Train:', result['train'].date.min().date(), 's.d.', result['train'].date.max().date(), '|', len(result['train']), 'observasi')
print('Test :', result['test'].date.min().date(), 's.d.', result['test'].date.max().date(), '|', len(result['test']), 'observasi')

Train: 2016-08-02 s.d. 2024-07-19 | 2076 observasi
Test : 2024-07-22 s.d. 2026-07-24 | 520 observasi


In [3]:
result['volatility_metrics']

In [4]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'volatility_forecasts.png')))

<IPython.core.display.Image object>


In [5]:
result['direction_metrics']

## Takeaways

- Gunakan QLIKE bersama MAE/RMSE: sebuah model bisa unggul pada satu loss dan tidak pada yang lain.
- Jangan menilai GARCH dengan directional accuracy; ia memodelkan varians bersyarat, bukan tanda return.
- Jika data BI-Rate, inflasi YoY, dan ID10Y yang bertanggal *availability* ditambahkan sesuai `data/raw/README.md`, pipeline akan melakukan as-of join dan memasukkan fitur tersebut hanya ke model arah.